# Session 4: Introduction to Supervised Learning — Classification vs. Generation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/buildLittleWorlds/level-2-course-material/blob/main/session-04/notebook.ipynb)

Tonight we crossed the fork in the road. For three weeks, every model we used did the same thing: **classification** — sorting text into buckets from a fixed menu. Tonight we met a model that does something fundamentally different: **generation** — creating new text, word by word.

In [ ]:
!pip install -q transformers torch
from transformers import pipeline
print("Setup complete!")

## The Fork in the Code

Here's the line that powered every Space in Sessions 1–3:

```python
analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
```

And here's tonight's line:

```python
generator = pipeline("text-generation", model="distilbert/distilgpt2")
```

Same library. Same function. Different task type. Let's load both and see the difference.

In [ ]:
# The OLD way: classification (Sessions 1-3)
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print("Classifier loaded (sorts into buckets)")

# The NEW way: generation (Session 4)
generator = pipeline("text-generation", model="distilbert/distilgpt2")
print("Generator loaded (creates new text)")

print("\nBoth models ready. Let's compare.")

---

## Side by Side: Classification vs. Generation

Feed the same text to both models. Watch how different the outputs are.

In [ ]:
text = "Monday morning arrived like a gift from the universe."

# Classification: picks a label
label_result = classifier(text)[0]
print("CLASSIFICATION (sorts into a bucket):")
print(f"  {label_result['label']} ({label_result['score']:.0%})")
print()

# Generation: writes new text
gen_result = generator(text, max_new_tokens=50, do_sample=True, truncation=True)
print("GENERATION (creates new text):")
print(f"  {gen_result[0]['generated_text']}")
print()
print("Same input. Completely different kinds of output.")

**What happened?** The classifier gave one word and a confidence number. The generator wrote a paragraph. One sorts. The other creates.

*Your observations:*



---

## Experiment 1: The Adversarial Stories

In Session 3, we broke classification models with three stories. What does the generator do with the same text?

In [ ]:
# The Sarcastic Narrator (Session 3: classifier missed the sarcasm)
sarcastic = "Her inbox had 47 unread emails, each one a little treasure waiting to be discovered."

print("THE SARCASTIC NARRATOR")
print(f'Input: "{sarcastic}"')
print()

label = classifier(sarcastic)[0]
print(f"Classifier says: {label['label']} ({label['score']:.0%})")
print("  (Last week: this was TONE DEAFNESS. The classifier missed the sarcasm.)")
print()

gen = generator(sarcastic, max_new_tokens=60, do_sample=True, truncation=True)
print(f"Generator writes: {gen[0]['generated_text']}")
print("  (Did the generator continue the sarcastic tone? Or lose it?)")

In [ ]:
# The Mixed-Emotion Story (Session 3: classifier flattened the emotions)
mixed = "That night, they celebrated with her favorite dinner, and nobody mentioned the empty chair that would be there next year."

print("THE MIXED-EMOTION STORY")
print(f'Input: "{mixed}"')
print()

label = classifier(mixed)[0]
print(f"Classifier says: {label['label']} ({label['score']:.0%})")
print("  (Last week: this was EMOTIONAL FLATTENING. The classifier picked one emotion.)")
print()

gen = generator(mixed, max_new_tokens=60, do_sample=True, truncation=True)
print(f"Generator writes: {gen[0]['generated_text']}")
print("  (Did the generator capture any of the emotional complexity?)")

In [ ]:
# The Nature Story (Session 3: classifier invented emotions)
nature = "The nearby forest burned for nine days, and then the rain came."

print("THE NATURE STORY")
print(f'Input: "{nature}"')
print()

label = classifier(nature)[0]
print(f"Classifier says: {label['label']} ({label['score']:.0%})")
print("  (Last week: this was ANTHROPOMORPHIC PROJECTION. The classifier invented emotions.)")
print()

gen = generator(nature, max_new_tokens=60, do_sample=True, truncation=True)
print(f"Generator writes: {gen[0]['generated_text']}")
print("  (Does the generator project human emotions onto nature too? Or something different?)")

**Reflect:** The classifier gives one label. The generator writes a continuation. Neither is "right" or "wrong" — they're doing completely different tasks. But which output gives you more information about the input?

*Your observations:*



---

## Experiment 2: Run It Twice

Classification always gives the same answer for the same input. Generation doesn't. Run the same prompt three times and see what happens.

In [ ]:
prompt = "Once upon a time, in a city made entirely of glass,"

print(f'Prompt: "{prompt}"')
print()

# Classification: same answer every time
for i in range(3):
    result = classifier(prompt)[0]
    print(f"Classifier run {i+1}: {result['label']} ({result['score']:.0%})")

print()

# Generation: different each time
for i in range(3):
    result = generator(prompt, max_new_tokens=40, do_sample=True, truncation=True)
    print(f"Generator run {i+1}: {result[0]['generated_text']}")
    print()

**What happened?** The classifier gave the same answer all three times. The generator gave three different continuations.

This is a fundamental difference. Classification is deterministic — same input, same label. Generation involves randomness — the model picks from a probability distribution over possible next words. That randomness is what **temperature** controls, and that's what we explore next week.

*Your observations:*



---

## Experiment 3: Your Own Prompts

Try your own text. Feed it to both models. See what each one does.

In [ ]:
# Replace this with anything you want to try!
my_text = "Replace this with your own sentence."

print(f'Your text: "{my_text}"')
print()

label = classifier(my_text)[0]
print(f"Classifier: {label['label']} ({label['score']:.0%})")
print()

gen = generator(my_text, max_new_tokens=60, do_sample=True, truncation=True)
print(f"Generator: {gen[0]['generated_text']}")

In [ ]:
# Try a few more. Ideas:
# - A simple fact ("The capital of France is")
# - Something creative ("In a world where cats could fly,")
# - Something from your interests
# - Something absurd

my_text_2 = "Replace this with another sentence."

label = classifier(my_text_2)[0]
gen = generator(my_text_2, max_new_tokens=60, do_sample=True, truncation=True)

print(f"Classifier: {label['label']} ({label['score']:.0%})")
print(f"Generator:  {gen[0]['generated_text']}")

---

## Experiment 4: How the Generator Picks Words

The generator doesn't just write the most likely next word every time. It has a probability for *every* possible next word. Let's peek at those probabilities.

In [ ]:
# This shows the top 10 most likely next words for a prompt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    next_token_logits = outputs.logits[0, -1, :]
    probs = torch.softmax(next_token_logits, dim=-1)

top_probs, top_indices = torch.topk(probs, 10)

print(f'Prompt: "{prompt}"')
print(f"\nTop 10 most likely next words:")
print("-" * 35)
for prob, idx in zip(top_probs, top_indices):
    token = tokenizer.decode(idx)
    print(f"  {token:>15s}  ({prob:.1%})")

print("\nThe model picks ONE of these (or any of the 50,000+ others).")
print("Next week: temperature controls HOW it picks.")

**What happened?** The model has a probability for every word in its vocabulary (∼50,000 words). It doesn't just "know" the answer — it has a spread of possibilities. Generation means *sampling* from that spread. Different samples, different outputs.

Next week, we learn that **temperature** controls how much the model concentrates on the top choices vs. spreading across the possibilities. Low temperature = safe and predictable. High temperature = creative and risky.

*Your observations:*



---

## The Fork: Classification vs. Generation

| | Classification | Generation |
|---|---|---|
| **What it does** | Sorts input into categories | Creates new content |
| **Output** | One label from a fixed menu | A sequence of new words |
| **Training data** | Input + label pairs (a human decided each label) | Just text — no labels needed |
| **Same input, multiple runs** | Always the same answer | Different each time |
| **The catch** | Labels are expensive (humans create them) | Text is everywhere (the internet) |

When you don't need labels, you can train on *everything*. That's the insight that led to ChatGPT, Claude, and every generative AI tool.

---

## Challenge

Find a prompt where the generator produces something *surprisingly good* AND a prompt where it produces something *clearly wrong or nonsensical*. What does that tell you about the strengths and limits of a small generation model?

---

**GitHub skill:** Upload this notebook to your `my-ai-portfolio` repo:
1. Go to your repo on github.com
2. Click **Add file** → **Upload files**
3. Drag your `.ipynb` file and click **Commit changes**

---

## Vocabulary

| Term | Meaning |
|------|--------|
| **Classification** | AI that sorts inputs into categories from a fixed menu |
| **Generation** | AI that creates new content by predicting the next word |
| **Labeled data** | Training data where a human decided the correct answer for each example |
| **Next-word prediction** | How generation models learn: read, predict, check, adjust |
| **Parameters** | Internal numbers a model adjusts during training (distilgpt2: 82M, GPT-4: 1T+) |
| **Sampling** | Choosing one word from the probability distribution over all possible next words |
| **Temperature** | A control that affects how the model samples (next week!) |